# AI Is Not Enough — Companion Code
### Solver-Verified Optimization Models for Chapters 8–18

**Author:** Dr. A.J. Klinkert  
**Repository:** [github.com/Klinkert/ai-is-not-enough-companion](https://github.com/Klinkert/ai-is-not-enough-companion)

---

This notebook contains the complete Python implementation for every optimization model in the book's Appendix (Chapters 8–18). Each section maps directly to its chapter. The models are solver-verified using Google OR-Tools.

### What these models demonstrate
Each chapter introduces a real-world decision problem where AI alone is insufficient. The code shows how Decision Intelligence — constrained optimization — produces feasible, optimal decisions that AI prediction cannot.

### How to use this notebook
1. Run **Cell 1** (Setup) once to install OR-Tools
2. Jump to any chapter section and run that cell independently
3. Modify the input parameters to explore how the optimal solution changes

---

| Chapter | Problem Type | Solver |
|---------|-------------|--------|
| 8  | Binary Knapsack / Lead Selection | SCIP |
| 9  | Linear Allocation — Marketing Budget | GLOP |
| 10 | Transportation Problem — Inventory | GLOP |
| 11 | Cost + Risk Tradeoff — Supplier Selection | GLOP |
| 12 | Shift Scheduling — Contact Center | CBC |
| 13 | Single Machine Scheduling | CBC |
| 14 | Vehicle Routing Problem (VRP) | OR-Tools Routing |
| 15 | Network Flow — Distribution Logistics | GLOP |
| 16 | Assignment Problem — Resource Assignment | CBC |
| 17 | Multi-Dimensional Assignment — Data Center | SCIP |
| 18 | Dynamic Re-Optimization — Closed-Loop AI+DI | SCIP |


## Setup — Install OR-Tools
Run this cell once. All subsequent cells depend on OR-Tools.

In [ ]:
import sys
!{sys.executable} -m pip install ortools

---
## Chapter 8 — Lead Prioritization
**Problem type:** Binary Knapsack  
**Decision:** Which leads to pursue given a fixed outreach time budget  
**Key insight:** Ranking leads by value alone ignores time constraints — the optimal set requires solving a binary selection problem

In [ ]:
from itertools import combinations
from ortools.linear_solver import pywraplp

leads    = ['A', 'B', 'C', 'D', 'E']
value    = {'A': 10, 'B': 7, 'C': 15, 'D': 6, 'E': 18}
time     = {'A': 2,  'B': 3, 'C': 5,  'D': 4, 'E': 6}
capacity = 10

# Brute force (ground truth for small instances)
best_value, best_set = 0, None
for r in range(len(leads) + 1):
    for subset in combinations(leads, r):
        if sum(time[i] for i in subset) <= capacity:
            v = sum(value[i] for i in subset)
            if v > best_value:
                best_value, best_set = v, subset
print("Brute Force:", best_set, "→ value", best_value)

# OR-Tools model
solver = pywraplp.Solver.CreateSolver("SCIP")
x = {i: solver.BoolVar(f"x_{i}") for i in leads}
solver.Maximize(sum(value[i] * x[i] for i in leads))
solver.Add(sum(time[i] * x[i] for i in leads) <= capacity)
solver.Solve()

selected    = [i for i in leads if x[i].solution_value() > 0.5]
total_value = sum(value[i] for i in selected)
total_time  = sum(time[i] for i in selected)
print("\nSolver Solution:")
print("Selected:", selected)
print("Total Value:", total_value, "| Time Used:", total_time, "/", capacity)

---
## Chapter 9 — Marketing Budget Allocation
**Problem type:** Linear Programming (LP)  
**Decision:** How to allocate a $100K budget across three channels  
**Key insight:** Optimal allocation depends on the full system of constraints, not individual channel rankings

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("GLOP")
x1 = solver.NumVar(10, solver.infinity(), "digital")
x2 = solver.NumVar(0, 60, "email")
x3 = solver.NumVar(0, 40, "events")

solver.Add(x1 + x2 + x3 == 100)
solver.Maximize(0.12 * x1 + 0.10 * x2 + 0.18 * x3)
solver.Solve()

print("Optimal Allocation:")
print(f"  Digital Ads : ${x1.solution_value():.0f}K")
print(f"  Email       : ${x2.solution_value():.0f}K")
print(f"  Events      : ${x3.solution_value():.0f}K")
print(f"  Total Return: {solver.Objective().Value():.2f}")

---
## Chapter 10 — Inventory Allocation
**Problem type:** Transportation Problem  
**Decision:** How much inventory to ship from each warehouse to each store  
**Key insight:** Optimal allocation must coordinate the full network simultaneously

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("GLOP")
x11 = solver.NumVar(0, solver.infinity(), "x11")
x12 = solver.NumVar(0, solver.infinity(), "x12")
x21 = solver.NumVar(0, solver.infinity(), "x21")
x22 = solver.NumVar(0, solver.infinity(), "x22")

solver.Maximize(8*x11 + 6*x12 + 7*x21 + 9*x22)
solver.Add(x11 + x12 <= 50)   # W1 supply
solver.Add(x21 + x22 <= 40)   # W2 supply
solver.Add(x11 + x21 <= 60)   # S1 demand
solver.Add(x12 + x22 <= 50)   # S2 demand
solver.Solve()

print("Optimal Allocation:")
print(f"  W1 → S1: {x11.solution_value():.0f}")
print(f"  W1 → S2: {x12.solution_value():.0f}")
print(f"  W2 → S1: {x21.solution_value():.0f}")
print(f"  W2 → S2: {x22.solution_value():.0f}")
print(f"  Total Value: {solver.Objective().Value():.0f}")

---
## Chapter 11 — Supplier Selection
**Problem type:** LP with Cost + Risk Tradeoff  
**Decision:** How many units to source from each supplier  
**Key insight:** Cost and risk must be balanced simultaneously under capacity constraints

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("GLOP")
x1 = solver.NumVar(0, 60, "x1")
x2 = solver.NumVar(0, 50, "x2")
x3 = solver.NumVar(0, 70, "x3")

solver.Add(x1 + x2 + x3 == 100)
solver.Add(0.2*x1 + 0.1*x2 + 0.3*x3 <= 25)
solver.Minimize(5*x1 + 6*x2 + 4*x3)
solver.Solve()

print("Optimal Sourcing:")
print(f"  Supplier 1: {x1.solution_value():.0f} units")
print(f"  Supplier 2: {x2.solution_value():.0f} units")
print(f"  Supplier 3: {x3.solution_value():.0f} units")
print(f"  Total Cost: {solver.Objective().Value():.0f}")

---
## Chapter 12 — Contact Center Staffing
**Problem type:** Integer Programming — Shift Scheduling  
**Decision:** How many agents to assign to each shift  
**Key insight:** Staffing is not a period-by-period decision — shifts span multiple periods and must be coordinated

In [ ]:
from ortools.linear_solver import pywraplp

solver  = pywraplp.Solver.CreateSolver("CBC")
periods = ["P1", "P2", "P3", "P4"]
demand  = {"P1": 3, "P2": 5, "P3": 4, "P4": 2}
shifts  = ["S1", "S2", "S3"]
cost    = {"S1": 8, "S2": 8, "S3": 8}
coverage = {
    ("S1","P1"): 1, ("S1","P2"): 1, ("S1","P3"): 0, ("S1","P4"): 0,
    ("S2","P1"): 0, ("S2","P2"): 1, ("S2","P3"): 1, ("S2","P4"): 0,
    ("S3","P1"): 0, ("S3","P2"): 0, ("S3","P3"): 1, ("S3","P4"): 1,
}
penalty = 100

x = {s: solver.IntVar(0, solver.infinity(), f"x_{s}") for s in shifts}
u = {p: solver.NumVar(0, solver.infinity(), f"u_{p}") for p in periods}

solver.Minimize(
    sum(cost[s]*x[s] for s in shifts) + penalty*sum(u[p] for p in periods)
)
for p in periods:
    solver.Add(sum(coverage[(s,p)]*x[s] for s in shifts) + u[p] >= demand[p])
solver.Solve()

print("Optimal Staffing:")
for s in shifts:
    print(f"  {s}: {int(x[s].solution_value())} agents")
print(f"  Total Cost: {solver.Objective().Value():.0f}")

---
## Chapter 13 — Production Scheduling
**Problem type:** Single Machine Scheduling (MIP)  
**Decision:** The sequence in which to process jobs to minimize total completion time  
**Key insight:** Sequencing requires binary precedence variables — you cannot sort your way to an optimal schedule

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("CBC")
jobs   = ["A", "B", "C"]
p      = {"A": 3, "B": 2, "C": 4}
M      = 100

C = {j: solver.NumVar(0, solver.infinity(), f"C_{j}") for j in jobs}
y = {(i,j): solver.IntVar(0, 1, f"y_{i}_{j}") for i in jobs for j in jobs if i != j}

for i in jobs:
    for j in jobs:
        if i < j:
            solver.Add(y[(i,j)] + y[(j,i)] == 1)
for i in jobs:
    for j in jobs:
        if i != j:
            solver.Add(C[j] >= C[i] + p[j] - M*(1 - y[(i,j)]))
for j in jobs:
    solver.Add(C[j] >= p[j])

solver.Minimize(sum(C[j] for j in jobs))
solver.Solve()

completion = {j: C[j].solution_value() for j in jobs}
sequence   = sorted(jobs, key=lambda j: completion[j])
print("Optimal Sequence:", " → ".join(sequence))
print("Completion Times:", {j: round(completion[j],1) for j in jobs})
print(f"Total Completion Time: {sum(completion.values()):.0f}")

---
## Chapter 14 — Field Service Routing
**Problem type:** Vehicle Routing Problem (VRP / TSP)  
**Decision:** The optimal route for a technician visiting multiple locations  
**Key insight:** Nearest-neighbor greedy routing is suboptimal — the full tour must be optimized globally

> **Scale note:** 10 stops → ~3.6M possible routes. 20 stops → infeasible to enumerate. OR-Tools routing handles real-world scale.

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

locations = ["Depot", "A", "B", "C", "D"]
distance_matrix = [
    [0, 4, 6, 8, 7],
    [4, 0, 2, 5, 6],
    [6, 2, 0, 4, 3],
    [8, 5, 4, 0, 2],
    [7, 6, 3, 2, 0],
]

manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    return distance_matrix[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(distance_callback))
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
solution = routing.SolveWithParameters(params)

index, route, total = routing.Start(0), [], 0
while not routing.IsEnd(index):
    route.append(locations[manager.IndexToNode(index)])
    prev, index = index, solution.Value(routing.NextVar(index))
    total += routing.GetArcCostForVehicle(prev, index, 0)
route.append("Depot")

print("Optimal Route:", " → ".join(route))
print(f"Total Travel Time: {total}")

---
## Chapter 15 — Distribution Logistics
**Problem type:** Network Flow  
**Decision:** How much product to send on each arc through a logistics network  
**Key insight:** Distribution must be modeled as a coordinated network, not as a collection of local shipment decisions

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("GLOP")
nodes  = ["Plant", "Hub", "Region1", "Region2"]
supply = {"Plant": 100, "Hub": 0, "Region1": -40, "Region2": -60}
arcs   = [("Plant","Hub"),("Plant","Region1"),("Hub","Region1"),("Hub","Region2")]
cost   = {("Plant","Hub"): 2, ("Plant","Region1"): 5, ("Hub","Region1"): 1, ("Hub","Region2"): 3}
cap    = {("Plant","Hub"): 100, ("Plant","Region1"): 40, ("Hub","Region1"): 60, ("Hub","Region2"): 80}

x = {(i,j): solver.NumVar(0, cap[(i,j)], f"x_{i}_{j}") for i,j in arcs}
solver.Minimize(sum(cost[(i,j)] * x[(i,j)] for i,j in arcs))
for n in nodes:
    solver.Add(sum(x[(i,j)] for i,j in arcs if i==n) - sum(x[(i,j)] for i,j in arcs if j==n) == supply[n])
solver.Solve()

print("Optimal Flows:")
for i,j in arcs:
    v = x[(i,j)].solution_value()
    if v > 1e-6:
        print(f"  {i} → {j}: {v:.0f} units")
print(f"Total Cost: {solver.Objective().Value():.0f}")

---
## Chapter 16 — Resource Assignment
**Problem type:** Assignment Problem (MIP)  
**Decision:** One-to-one matching of technicians to jobs at minimum cost  
**Key insight:** A low-cost match in isolation may block a better overall assignment — only global optimization finds the true optimum

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("CBC")
techs  = ["A", "B", "C"]
jobs   = [1, 2, 3]
cost   = {("A",1):9,("A",2):2,("A",3):7,("B",1):6,("B",2):4,("B",3):3,("C",1):5,("C",2):8,("C",3):1}

x = {(i,j): solver.IntVar(0, 1, f"x_{i}_{j}") for i in techs for j in jobs}
for i in techs:
    solver.Add(sum(x[(i,j)] for j in jobs) == 1)
for j in jobs:
    solver.Add(sum(x[(i,j)] for i in techs) == 1)
solver.Minimize(sum(cost[(i,j)] * x[(i,j)] for i in techs for j in jobs))
solver.Solve()

print("Optimal Assignment:")
for i in techs:
    for j in jobs:
        if x[(i,j)].solution_value() > 0.5:
            print(f"  Tech {i} → Job {j}  (cost={cost[(i,j)]})")
print(f"Total Cost: {sum(cost[(i,j)] for i in techs for j in jobs if x[(i,j)].solution_value()>0.5):.0f}")

---
## Chapter 17 — Data Center Resource Allocation
**Problem type:** Multi-Dimensional Knapsack (MIP)  
**Decision:** Which workloads to place on which servers, subject to CPU and memory limits  
**Key insight:** A workload feasible in CPU may fail in memory — placement requires coordinated multi-resource optimization

In [ ]:
from ortools.linear_solver import pywraplp

solver    = pywraplp.Solver.CreateSolver("SCIP")
workloads = ["W1","W2","W3","W4"]
servers   = ["S1","S2"]
value     = {"W1":10,"W2":8,"W3":7,"W4":6}
cpu       = {"W1":4,"W2":3,"W3":2,"W4":3}
mem       = {"W1":6,"W2":4,"W3":3,"W4":2}
cpu_cap   = {"S1":6,"S2":6}
mem_cap   = {"S1":8,"S2":7}

x = {(w,s): solver.BoolVar(f"x_{w}_{s}") for w in workloads for s in servers}
solver.Maximize(sum(value[w]*x[(w,s)] for w in workloads for s in servers))
for w in workloads:
    solver.Add(sum(x[(w,s)] for s in servers) <= 1)
for s in servers:
    solver.Add(sum(cpu[w]*x[(w,s)] for w in workloads) <= cpu_cap[s])
    solver.Add(sum(mem[w]*x[(w,s)] for w in workloads) <= mem_cap[s])
solver.Solve()

print("Optimal Placement:")
total = 0
for s in servers:
    assigned = [w for w in workloads if x[(w,s)].solution_value() > 0.5]
    total += sum(value[w] for w in assigned)
    print(f"  {s}: {assigned}")
print(f"Total Value: {total}")

---
## Chapter 18 — Dynamic Re-Optimization
**Problem type:** Closed-Loop AI + DI  
**Decision:** Re-solve the placement model when AI detects changed conditions  
**Key insight:** The model structure does not change — only the inputs change. AI updates the state; DI re-solves the decision.

In [ ]:
from ortools.linear_solver import pywraplp

def solve_placement(value_w3, s2_mem_cap):
    solver    = pywraplp.Solver.CreateSolver("SCIP")
    workloads = ["W1","W2","W3","W4"]
    servers   = ["S1","S2"]
    value     = {"W1":10,"W2":8,"W3":value_w3,"W4":6}
    cpu       = {"W1":4,"W2":3,"W3":2,"W4":3}
    mem       = {"W1":6,"W2":4,"W3":3,"W4":2}
    cpu_cap   = {"S1":6,"S2":6}
    mem_cap   = {"S1":8,"S2":s2_mem_cap}

    x = {(w,s): solver.BoolVar(f"x_{w}_{s}") for w in workloads for s in servers}
    solver.Maximize(sum(value[w]*x[(w,s)] for w in workloads for s in servers))
    for w in workloads:
        solver.Add(sum(x[(w,s)] for s in servers) <= 1)
    for s in servers:
        solver.Add(sum(cpu[w]*x[(w,s)] for w in workloads) <= cpu_cap[s])
        solver.Add(sum(mem[w]*x[(w,s)] for w in workloads) <= mem_cap[s])
    solver.Solve()

    assignment   = {s: [w for w in workloads if x[(w,s)].solution_value()>0.5] for s in servers}
    total_value  = sum(value[w]*x[(w,s)].solution_value() for w in workloads for s in servers)
    return assignment, total_value

# Initial state
before, v_before = solve_placement(value_w3=7,  s2_mem_cap=7)
# AI detects: W3 value increases, S2 memory drops
after,  v_after  = solve_placement(value_w3=11, s2_mem_cap=6)

print("Before (AI update):")
print(f"  {before}  →  Total value: {v_before:.0f}")
print("\nAfter (DI re-solves):")
print(f"  {after}  →  Total value: {v_after:.0f}")